# ADV head motion
VB5, VE4 and VE7 have loose probe heads (`sensor_notes.csv`). These are cable heads: the compass and
tilt sensor are in the canister, so nothing measured the head's own attitude, and `adv_QC` section 5
rotates the whole record with one KVH heading. This notebook finds **when** a head turned or tilted,
and by how much, from the velocity data themselves. It only detects and flags. It writes no corrected
velocities and does not touch `{S}_QC.nc`; how to handle each event (truncate, split headings, flag)
is decided separately.

Three signatures, all from the wave band (0.05-0.25 Hz), per window on a common UTC grid:
1. **Head rotation about the vertical.** The wave direction of travel from the p-u, p-v co-spectra
   is compared with fixed-head references (VC5, VE10). Refraction makes the offset between sensors
   non-zero, but it is steady; a **step** in the offset is the head turning.
   Sign: if the head turns **+δ (clockwise seen from above)** and is still rotated with the deployment
   KVH heading, every measured direction comes out δ too small, so Δθ steps by **−δ**. The tables
   report `head_rot_deg = −step`. This sign follows from the geometry; the synthetic test (2a) injects
   a rotation under the same assumption, so it checks the timing and size of a recovered step, not
   the sign. An independent sign check is the +X arm bearing at recovery (photos / diver KVH), which
   should read about KVH + the last segment's `head_rot_vs_start_deg`.
2. **Head tilt.** For linear waves the true w is 90° out of phase with u, so the part of w that is
   *in phase* with the horizontal velocity is leakage from a tilted head (plus a constant from the bed
   slope). A least-squares fit of Co(w,E), Co(w,N) on the horizontal co-spectra gives a tilt
   magnitude and the bearing the head leans toward. A **step** in it is the head tilting.
3. **Disturbance.** w-dp/dt coherence drops when the head tilts or is knocked about. Stretches are
   flagged when the Δθ jitter is high (head moving) or when coherence sits below the references'.

Input: `data/processed/qc/{S}_QC.nc` for the target and the references (read-only).
Output: `data/processed/qc/headmotion/{S}_headmotion_events.csv`, `..._segments.csv`, and
`figs/qc/{S}_headmotion*.png`.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
from scipy.signal import detrend, find_peaks

### 0. Sensor selection and parameters
Change `sensor_id` and run all. The batch cell at the end runs every sensor in `TARGETS` and keeps an
executed copy of each in `processing/qc_runs/`.

In [ ]:
##-----sensor selection-----##
sensor_id = "VE4"
TARGETS   = ["VB5", "VE4", "VE7"]        # loose heads (sensor_notes.csv), run by the batch cell

##-----references-----##
# Fixed-head sensors whose wave direction is the yardstick. VC10 is left out: its wave direction
# steps ~+10 deg on 08-28 against the ADCP (cause unknown). A target is never its own reference.
REFS         = ["VC5", "VE10"]
CONTROL_PAIR = ("VC5", "VE10")           # fixed vs fixed: sets the natural-variability thresholds

##-----windows-----##
WIN_S        = 1024     # s. Coarse window for the overview and step search (as the Z-test segments)
FINE_WIN_S   = 300      # s. Fine window used to pin down the time of each step
BAND         = (0.05, 0.25)   # Hz, wave band for every co-spectrum
NAN_MAX_FRAC = 0.10     # windows with more NaN (any of p, u, v, w) are skipped; the rest are filled linearly
DIR_R_MIN    = 0.5      # a reference window's direction is used only if its p-u_h coherence >= this
                        # (low = waves too small or too spread for a stable direction)

##-----step detection-----##
STEP_HALF_H    = 12     # h. The step statistic compares the next 12 h with the previous 12 h
STEP_FLOOR_DEG = 5.0    # deg. Smallest heading step reported, whatever the control says
TILT_FLOOR_DEG = 2.0    # deg. Smallest tilt step reported
STEP_K         = 1.5    # threshold = max(floor, STEP_K * 99th percentile of |step statistic| on the control pair)

##-----disturbed stretches-----##
JITTER_WIN_H  = 6       # h. Rolling window for the Δθ jitter and the coherence deficit
JITTER_K      = 2.5     # unsteady if jitter > JITTER_K * the target's own quiet level (25th percentile)
COH_DROP      = 0.15    # low_coh if (ref coherence - target coherence) exceeds its quiet level by this
MIN_STRETCH_H = 3       # h. Shorter flagged runs are dropped
MERGE_GAP_H   = 2       # h. Flagged runs closer than this are merged

##-----figures / checks-----##
ZOOM_H        = 12      # h either side of an event in the zoom figures
RUN_SYNTHETIC = True    # section 2: inject a known head rotation into VC5 and check it is recovered
INJECT        = ("2026-08-10 00:00", 10.0)   # (time, head rotation deg, + clockwise from above)

RHO, GRAV = 1023.0, 9.79                    # kg/m3, m/s2 (as adv_QC)
EPOCH     = np.datetime64("2026-07-01T00:00:00")   # common window grid origin for every sensor

# paths
data_path = "/Users/isidorarojas/Desktop/DIRECTORY/2026/4EWA/data/processed"
root    = Path(data_path).parents[1]
qc_dir  = Path(data_path) / "qc"
out_dir = qc_dir / "headmotion"
fig_dir = root / "figs" / "qc"
out_dir.mkdir(exist_ok=True)

# figure style, same palette as adv_QC
INK, INK2, GRID, SERIES = "#0b0b0b", "#52514e", "#e4e3df", "#2a78d6"
BEAM_COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]   # beams 1, 2, 3 (fixed order)
REF_COLORS  = ["#8a8883", "#b9b7b1"]              # references, grays so the target stands out
SHADE = {"unsteady": "#f3c7ae", "low_coh": "#d6d3f0"}   # disturbed-stretch shading
plt.rcParams.update({
    "axes.edgecolor": INK2, "axes.labelcolor": INK2, "xtick.color": INK2,
    "ytick.color": INK2, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "figure.facecolor": "white", "axes.facecolor": "white", "font.size": 9,
})

### 1. Windowed diagnostics
`load_sensor` reads only the variables needed (no `.load()` of the whole file). Each sample is placed
on a regular 2 Hz grid counted from `EPOCH`, so window *n* covers the same UTC span in every sensor and
target/reference windows pair up one to one. The clock-drift correction made the spacing slightly
irregular; rounding to the grid moves a handful of samples by one slot, and the few duplicates or holes
this leaves are handled like any other NaN.

Per window (skipped if any of p, u, v, w is > `NAN_MAX_FRAC` NaN; else linear fill, linear detrend,
Hann taper, one FFT):
- `wave_dir`: direction of travel, atan2(Co(E,p), Co(N,p)), deg true (as adv_QC section 5)
- `r_puv`: |Co(p,u_h)| / sqrt(Spp (SEE+SNN)), how well defined that direction is (1 = unidirectional)
- `coh_w_dpdt`: as adv_QC (≈ +1 for a vertical, head-up probe)
- `tilt_deg`, `tilt_bearing`: in-phase leakage of horizontal velocity into w,
  [Co(w,E), Co(w,N)] = [[SEE, CEN], [CEN, SNN]] · [a, b]; tilt = atan(√(a²+b²)), leaning toward atan2(a, b)
- `Hs_p`: 4·std of bed pressure in the wave band, m of water, **not** depth-corrected (an energy proxy)
- on every window: `gated_pct` (% failing the correlation gate), median corr/amp per beam, canister
  heading (circular mean), pitch, roll

In [ ]:
def load_sensor(S):
    """The variables this notebook needs from {S}_QC.nc, plus each sample's index on the EPOCH grid."""
    ds = xr.open_dataset(qc_dir / f"{S}_QC.nc")
    fs = float(ds.attrs["fs"])
    dt = np.timedelta64(int(round(1e9 / fs)), "ns")
    d = {"S": S, "fs": fs, "t": ds.time.values}
    d["k"] = np.round((d["t"] - EPOCH) / dt).astype(np.int64)
    for v in ["p", "u_east", "v_north", "w", "heading", "pitch", "roll"]:
        d[v] = ds[v].values
    d["corr"] = ds["corr"].values            # (beam, time), %
    d["amp"]  = ds["amp"].values             # (beam, time), counts
    d["gated"] = ~ds["vel_corr_ok"].values   # True = velocity NaN'd by the correlation gate
    d["kvh"] = float(ds.attrs["rot_heading_kvh_magnetic_deg"])
    ds.close()
    return d

def to_windows(d, x, N):
    """(n_win, N) array of x on the EPOCH grid, NaN where no sample landed. Also returns the first window number."""
    w0 = d["k"][0] // N
    nw = d["k"][-1] // N + 1 - w0
    a = np.full(nw * N, np.nan, np.float32)
    a[d["k"] - w0 * N] = x
    return a.reshape(nw, N), w0

def window_diag(d, win_s, band=BAND, inject=None):
    """Per-window wave-band diagnostics, indexed by window start time (UTC).
    inject = (time, deg): from that time on, rotate E/N as if the head had turned deg clockwise."""
    fs = d["fs"]
    N = int(round(win_s * fs))
    E, Nv = d["u_east"].astype(float), d["v_north"].astype(float)
    if inject is not None:
        # head +delta clockwise -> reported bearings are delta too small -> rotate the vector by -delta
        m = d["t"] >= np.datetime64(pd.Timestamp(inject[0]))
        phi = np.radians(-inject[1])
        E, Nv = (np.where(m, E * np.cos(phi) + Nv * np.sin(phi), E),
                 np.where(m, -E * np.sin(phi) + Nv * np.cos(phi), Nv))
    W = {}
    for c, x in {"p": d["p"], "E": E, "N": Nv, "w": d["w"]}.items():
        W[c], w0 = to_windows(d, x, N)
    nw = W["p"].shape[0]
    idx = pd.DatetimeIndex(EPOCH + np.arange(w0, w0 + nw) * N * np.timedelta64(int(round(1e9 / fs)), "ns"),
                           name="time")
    out = pd.DataFrame(index=idx)
    out["nan_frac"] = np.max([np.isnan(W[c]).mean(1) for c in W], 0)
    ok = out["nan_frac"].to_numpy() <= NAN_MAX_FRAC

    f = np.fft.rfftfreq(N, 1 / fs)
    b = (f >= band[0]) & (f <= band[1])
    taper = np.hanning(N)
    F = {}
    for c in W:
        X = pd.DataFrame(W[c][ok].astype(float)).interpolate(axis=1, limit_direction="both").to_numpy()
        F[c] = np.fft.rfft(detrend(X, axis=1) * taper, axis=1)[:, b]
    co = lambda a, c: np.real(np.sum(F[a] * np.conj(F[c]), 1))
    Spp, SEE, SNN, CEN = co("p", "p"), co("E", "E"), co("N", "N"), co("E", "N")
    CpE, CpN, CwE, CwN = co("E", "p"), co("N", "p"), co("w", "E"), co("w", "N")
    Pdot = F["p"] * 1j * 2 * np.pi * f[b]
    coh = np.real(np.sum(F["w"] * np.conj(Pdot), 1)) / np.sqrt(np.sum(abs(F["w"]) ** 2, 1) * np.sum(abs(Pdot) ** 2, 1))
    det = SEE * SNN - CEN ** 2
    ta = (CwE * SNN - CwN * CEN) / det             # w leakage per unit E velocity
    tb = (CwN * SEE - CwE * CEN) / det             # w leakage per unit N velocity
    var_p = 2 * Spp / (N * np.sum(taper ** 2))     # wave-band variance of p, dbar^2 (Parseval, Hann-corrected)
    res = {"wave_dir": np.degrees(np.arctan2(CpE, CpN)) % 360,
           "r_puv": np.hypot(CpE, CpN) / np.sqrt(Spp * (SEE + SNN)),
           "coh_w_dpdt": coh,
           "tilt_deg": np.degrees(np.arctan(np.hypot(ta, tb))),
           "tilt_bearing": np.degrees(np.arctan2(ta, tb)) % 360,
           "Hs_p": 4 * np.sqrt(var_p) * 1e4 / (RHO * GRAV)}
    for name, v in res.items():
        col = np.full(nw, np.nan)
        col[ok] = v
        out[name] = col

    # beam quality and canister attitude, on every window (no NaN rule)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)          # all-NaN windows in record gaps
        out["gated_pct"] = 100 * np.nanmean(to_windows(d, d["gated"].astype(np.float32), N)[0], 1)
        for i in range(3):
            out[f"corr{i + 1}"] = np.nanmedian(to_windows(d, d["corr"][i].astype(np.float32), N)[0], 1)
            out[f"amp{i + 1}"]  = np.nanmedian(to_windows(d, d["amp"][i].astype(np.float32), N)[0], 1)
        hr = np.radians(to_windows(d, d["heading"], N)[0])
        out["can_heading"] = np.degrees(np.arctan2(np.nanmean(np.sin(hr), 1), np.nanmean(np.cos(hr), 1))) % 360
        out["can_pitch"] = np.nanmedian(to_windows(d, d["pitch"], N)[0], 1)
        out["can_roll"]  = np.nanmedian(to_windows(d, d["roll"], N)[0], 1)
    return out.dropna(how="all", subset=["gated_pct"])

In [ ]:
##-----diagnostics for the target and the references-----##
refs = [r for r in REFS if r != sensor_id]
diag = {}            # diag[S] = {"c": coarse DataFrame, "f": fine DataFrame}
raw = None           # the target's 2 Hz arrays, kept for the zoom figures
for S in dict.fromkeys([sensor_id, *refs, *CONTROL_PAIR]):
    d = load_sensor(S)
    diag[S] = {"c": window_diag(d, WIN_S), "f": window_diag(d, FINE_WIN_S)}
    if S == sensor_id:
        raw = d
    elif RUN_SYNTHETIC and S == CONTROL_PAIR[0]:
        # section 2 needs this sensor with a known head rotation injected
        diag["synthetic"] = {"c": window_diag(d, WIN_S, inject=INJECT), "f": window_diag(d, FINE_WIN_S, inject=INJECT)}
    if S != sensor_id:
        del d
    c = diag[S]["c"]
    print(f"{S}: {c.wave_dir.notna().sum()} of {len(c)} {WIN_S}-s windows usable "
          f"({c.index[0]:%m-%d %H:%M} -> {c.index[-1]:%m-%d %H:%M}); median coh_w_dpdt {c.coh_w_dpdt.median():+.2f}, "
          f"tilt {c.tilt_deg.median():.1f} deg, r_puv {c.r_puv.median():.2f}")

### 2. Rotation signal, thresholds, and the detector
- **Δθ** = target wave direction − circular mean of the references (only reference windows with
  `r_puv ≥ DIR_R_MIN`), wrapped to ±180°.
- **Step statistic** s(t) = robust circular location of Δθ over the next `STEP_HALF_H` hours minus that
  over the previous `STEP_HALF_H` hours (location = atan2 of the rolling medians of sin and cos, so a
  few bad windows don't move it). A head that turns once gives a single peak in |s| at the turn.
- **Thresholds** come from the control pair (two fixed heads): the same statistic on VC5 − VE10 shows
  how far refraction and changing swell alone move it. Steps are reported above
  `max(STEP_FLOOR_DEG, STEP_K × 99th percentile)`; the same for tilt.
- Each candidate is **refined** on the `FINE_WIN_S` windows within ±`STEP_HALF_H`: the single split
  point that minimises the absolute deviation from each side's location. The event time is the start
  of the first fine window after the split (± one fine window).

In [ ]:
def wrap(a):
    return (a + 180) % 360 - 180

def circ_loc(a):
    """Robust circular location (deg): atan2 of the medians of sin and cos. NaN ignored."""
    r = np.radians(np.asarray(a, float))
    return np.degrees(np.arctan2(np.nanmedian(np.sin(r)), np.nanmedian(np.cos(r))))

def regular(s, win_s):
    """Reindex a window series onto the full regular grid (record gaps become NaN rows)."""
    return s.reindex(pd.date_range(s.index[0], s.index[-1], freq=f"{win_s}s"))

def ref_direction(ref_names, key):
    """Circular mean of the reference wave directions per window (windows with r_puv < DIR_R_MIN dropped)."""
    cols = [diag[r][key].wave_dir.where(diag[r][key].r_puv >= DIR_R_MIN).rename(r) for r in ref_names]
    D = pd.concat(cols, axis=1)
    rr = np.radians(D)
    m = np.degrees(np.arctan2(np.sin(rr).sum(1, min_count=1), np.cos(rr).sum(1, min_count=1))) % 360
    return m, D

def delta_theta(tgt, ref_names, key, win_s):
    m, D = ref_direction(ref_names, key)
    dth = regular(wrap(tgt.wave_dir - m.reindex(tgt.index)), win_s)
    per_ref = {r: regular(wrap(tgt.wave_dir - D[r].reindex(tgt.index)), win_s) for r in ref_names}
    return dth, per_ref

def step_stat(x, n, circular=True):
    """Value at t = location over [t, t+n) minus location over [t-n, t). NaN-tolerant (min 1/3 of n)."""
    minp = max(3, n // 3)
    if not circular:
        before = x.rolling(n, min_periods=minp).median().shift(1)
        after = x[::-1].rolling(n, min_periods=minp).median()[::-1]
        return after - before
    r = np.radians(x)
    s, c = np.sin(r), np.cos(r)
    loc_b = np.degrees(np.arctan2(s.rolling(n, min_periods=minp).median(), c.rolling(n, min_periods=minp).median())).shift(1)
    loc_a = np.degrees(np.arctan2(s[::-1].rolling(n, min_periods=minp).median(), c[::-1].rolling(n, min_periods=minp).median()))[::-1]
    return wrap(loc_a - loc_b)

def refine_step(xf, t_c, circular=True):
    """Single split of the fine series within +-STEP_HALF_H of t_c minimising the summed absolute deviation.
    Returns (event time, location before, location after)."""
    half = pd.Timedelta(hours=STEP_HALF_H)
    seg = xf[(xf.index >= t_c - half) & (xf.index < t_c + half)].dropna()
    if len(seg) < 10:
        return t_c, np.nan, np.nan
    v = seg.to_numpy()
    loc = circ_loc if circular else np.nanmedian
    dev = (lambda a, m: np.abs(wrap(a - m))) if circular else (lambda a, m: np.abs(a - m))
    costs = [dev(v[:j], loc(v[:j])).sum() + dev(v[j:], loc(v[j:])).sum() for j in range(3, len(v) - 3)]
    j = 3 + int(np.argmin(costs))
    return seg.index[j], loc(v[:j]), loc(v[j:])

def detect_steps(xc, xf, thr, win_s, circular=True):
    """Candidate steps from the coarse step statistic, refined on the fine series."""
    n = int(round(STEP_HALF_H * 3600 / win_s))
    s = step_stat(xc, n, circular)
    pk, _ = find_peaks(s.abs().fillna(0).to_numpy(), height=thr, distance=n)
    rows = []
    for t_c in s.index[pk]:
        t_e, before, after = refine_step(xf, t_c, circular)
        size = wrap(after - before) if circular else after - before
        if abs(size) >= thr:            # the refined step must still clear the threshold
            rows.append({"time": t_e, "coarse_time": t_c, "before": before, "after": after, "step": size})
    return s, pd.DataFrame(rows, columns=["time", "coarse_time", "before", "after", "step"])

##-----control pair: natural variability -> thresholds-----##
nC = int(round(STEP_HALF_H * 3600 / WIN_S))
cA, cB = (diag[S]["c"] for S in CONTROL_PAIR)
ctrl_dth = regular(wrap(cA.wave_dir.where(cA.r_puv >= DIR_R_MIN)
                        - cB.wave_dir.where(cB.r_puv >= DIR_R_MIN).reindex(cA.index)), WIN_S)
ctrl_s = step_stat(ctrl_dth, nC)
STEP_THR = max(STEP_FLOOR_DEG, STEP_K * np.nanpercentile(ctrl_s.abs(), 99))
ctrl_tilt_s = pd.concat([step_stat(regular(diag[S]["c"].tilt_deg, WIN_S), nC, circular=False) for S in CONTROL_PAIR])
TILT_THR = max(TILT_FLOOR_DEG, STEP_K * np.nanpercentile(ctrl_tilt_s.abs(), 99))
print(f"control {CONTROL_PAIR[0]} - {CONTROL_PAIR[1]}: Δθ median {np.nanmedian(ctrl_dth):+.1f} deg, "
      f"|step statistic| 99th pct {np.nanpercentile(ctrl_s.abs(), 99):.1f}, max {ctrl_s.abs().max():.1f} deg")
print(f"  -> heading-step threshold {STEP_THR:.1f} deg; tilt-step threshold {TILT_THR:.1f} deg "
      f"(control tilt |step| 99th pct {np.nanpercentile(ctrl_tilt_s.abs(), 99):.1f})")

#### 2a. Synthetic check
The control-pair sensor (VC5) gets a known head rotation from `INJECT` on (its E/N vectors are turned
as a head turned clockwise by that angle would make them), and goes through the same detector with the
other control sensor as the only reference. It must find one step at the right time (± one coarse
window) with `head_rot_deg` within ±2° of the injected angle (step sizes are good to about ±2°, since
the references' own swell-driven wander is still in Δθ).

In [ ]:
if RUN_SYNTHETIC:
    t_inj, d_inj = pd.Timestamp(INJECT[0]), INJECT[1]
    syn = diag["synthetic"]
    m_c, _ = ref_direction([CONTROL_PAIR[1]], "c")
    m_f, _ = ref_direction([CONTROL_PAIR[1]], "f")
    xc = regular(wrap(syn["c"].wave_dir.where(syn["c"].r_puv >= DIR_R_MIN) - m_c.reindex(syn["c"].index)), WIN_S)
    xf = regular(wrap(syn["f"].wave_dir.where(syn["f"].r_puv >= DIR_R_MIN) - m_f.reindex(syn["f"].index)), FINE_WIN_S)
    _, ev_syn = detect_steps(xc, xf, STEP_THR, WIN_S)
    print(f"injected head rotation {d_inj:+.1f} deg at {t_inj}:")
    print(ev_syn.assign(head_rot_deg=-ev_syn.step).round(2).to_string(index=False))
    assert len(ev_syn) == 1, f"expected exactly one step, found {len(ev_syn)}"
    dt_err = abs(ev_syn.time[0] - t_inj)
    assert dt_err <= pd.Timedelta(seconds=WIN_S), f"step time off by {dt_err}"
    assert abs(-ev_syn.step[0] - d_inj) <= 2, f"head rotation {-ev_syn.step[0]:+.1f}, expected {d_inj:+.1f}"
    print(f"synthetic check passed: time error {dt_err}, head_rot {-ev_syn.step[0]:+.1f} deg")

### 3. Detection on the target
1. **Heading steps** on Δθ (target vs the mean of the references).
2. **Tilt steps** on `tilt_deg`. The tilt estimate also contains the bed slope (a constant) and an
   in-phase w-u part from wave nonlinearity that grows with wave height, so check the tilt-vs-Hs
   figure before trusting a tilt step that coincides with a swell event.
3. **Disturbed stretches**, on the coarse grid, over `JITTER_WIN_H` rolling windows:
   - `unsteady`: rolling median of |ΔΔθ| between consecutive windows > `JITTER_K` × the target's own
     quiet level (25th percentile). Jitter, not the rolling spread, so a single clean step does not
     register as a stretch.
   - `low_coh`: (reference coherence − target coherence) > its quiet level (25th percentile) + `COH_DROP`.
     A tilted or obstructed head lowers coherence even when its direction is steady. `low_coh` therefore
     covers both an active disturbance and a head that has settled at a new tilt; the head *moving*
     shows as steps and `unsteady` stretches.

**Known false positive (null check on fixed heads VA10, VD5, VD10, 2026-10-03):** during the 09-08
swell peak VA10 and VD10 show apparent heading steps of 14° and 9°. Any target step at the same
time is suspect unless it persists after the swell and comes with a coherence/tilt change. Run a
fixed head as `sensor_id` to check any other date. VB5 (transect B) is compared with refs on C and
E, whose refraction can differ.

In [ ]:
tc, tf = diag[sensor_id]["c"], diag[sensor_id]["f"]
dth_c, dth_c_ref = delta_theta(tc, refs, "c", WIN_S)
dth_f, _ = delta_theta(tf, refs, "f", FINE_WIN_S)
s_head, ev_head = detect_steps(dth_c, dth_f, STEP_THR, WIN_S)
tilt_c, tilt_f = regular(tc.tilt_deg, WIN_S), regular(tf.tilt_deg, FINE_WIN_S)
s_tilt, ev_tilt = detect_steps(tilt_c, tilt_f, TILT_THR, WIN_S, circular=False)

##-----disturbed stretches-----##
nJ = int(round(JITTER_WIN_H * 3600 / WIN_S))
jitter = wrap(dth_c.diff()).abs().rolling(nJ, min_periods=nJ // 3, center=True).median()
jit_quiet = np.nanpercentile(jitter, 25)
coh_c = regular(tc.coh_w_dpdt, WIN_S)
ref_coh = pd.concat([regular(diag[r]["c"].coh_w_dpdt, WIN_S) for r in refs], axis=1).mean(1).reindex(coh_c.index)
deficit = (ref_coh - coh_c).rolling(nJ, min_periods=nJ // 3, center=True).median()
def_quiet = np.nanpercentile(deficit, 25)
flags = pd.DataFrame({"unsteady": (jitter > JITTER_K * jit_quiet).to_numpy(),
                      "low_coh": (deficit > def_quiet + COH_DROP).to_numpy()}, index=dth_c.index)

def runs(mask):
    """(start, stop) index pairs of True runs, stop inclusive (as adv_QC)."""
    e = np.flatnonzero(np.diff(np.r_[0, mask.astype(int), 0]))
    return e[::2], e[1::2] - 1

def stretches(mask, idx, win_s):
    """Flagged runs, gaps < MERGE_GAP_H merged, runs < MIN_STRETCH_H dropped -> list of (start, end)."""
    m = mask.copy()
    gap = int(round(MERGE_GAP_H * 3600 / win_s))
    st, sp = runs(~m)
    for a, b in zip(st, sp):                       # fill short False gaps between two flagged runs
        if a > 0 and b < len(m) - 1 and b - a + 1 <= gap:
            m[a:b + 1] = True
    st, sp = runs(m)
    keep = (sp - st + 1) * win_s >= MIN_STRETCH_H * 3600
    return [(idx[a], idx[b] + pd.Timedelta(seconds=win_s)) for a, b in zip(st[keep], sp[keep])]

stretch = {k: stretches(flags[k].to_numpy(), flags.index, WIN_S) for k in ["unsteady", "low_coh"]}

print(f"{sensor_id} vs refs {refs}: Δθ median {np.nanmedian(dth_c):+.1f} deg")
print(f"  quiet levels: jitter {jit_quiet:.2f} deg (flag > {JITTER_K * jit_quiet:.2f}); "
      f"coherence deficit {def_quiet:+.2f} (flag > {def_quiet + COH_DROP:+.2f})")
print(f"  heading steps (head_rot = -step, + clockwise from above):")
print(ev_head.assign(head_rot_deg=-ev_head.step).round(1).to_string(index=False) if len(ev_head) else "    none")
print(f"  tilt steps:")
print(ev_tilt.round(2).to_string(index=False) if len(ev_tilt) else "    none")
for k, v in stretch.items():
    print(f"  {k} stretches: {len(v)}")
    for a, b in v:
        print(f"    {a:%m-%d %H:%M} -> {b:%m-%d %H:%M} ({(b - a) / pd.Timedelta(hours=1):.0f} h)")

### 4. Events and segments
- **Events** (`{S}_headmotion_events.csv`): one row per heading step, tilt step, or disturbed stretch.
  For steps: Δθ, tilt and coherence over `STEP_HALF_H` before/after (fine windows).
- **Segments** (`{S}_headmotion_segments.csv`): the record split at every event boundary. Each segment
  is `stable`, `unsteady` or `low_coh` (majority of its windows), with its median Δθ, the implied head
  rotation since the start of the record (`head_rot_vs_start_deg` = −(Δθ − Δθ of the first segment),
  + clockwise from above; this assumes the head was at its KVH heading in the first segment), and its
  median tilt and coherence.

In [ ]:
half = pd.Timedelta(hours=STEP_HALF_H)
def side_stats(t):
    b = (tf.index >= t - half) & (tf.index < t)
    a = (tf.index >= t) & (tf.index < t + half)
    db, da = dth_f.reindex(tf.index)[b], dth_f.reindex(tf.index)[a]
    return {"dtheta_before": circ_loc(db), "dtheta_after": circ_loc(da),
            "tilt_before": tf.tilt_deg[b].median(), "tilt_after": tf.tilt_deg[a].median(),
            "coh_before": tf.coh_w_dpdt[b].median(), "coh_after": tf.coh_w_dpdt[a].median()}

rows = []
for _, e in ev_head.iterrows():
    rows.append({"sensor": sensor_id, "type": "heading_step", "time": e.time, "end": pd.NaT,
                 "size": e.step, "head_rot_deg": -e.step, "time_unc_s": FINE_WIN_S, **side_stats(e.time)})
for _, e in ev_tilt.iterrows():
    rows.append({"sensor": sensor_id, "type": "tilt_step", "time": e.time, "end": pd.NaT,
                 "size": e.step, "head_rot_deg": np.nan, "time_unc_s": FINE_WIN_S, **side_stats(e.time)})
for k, v in stretch.items():
    for a, b in v:
        rows.append({"sensor": sensor_id, "type": k, "time": a, "end": b, "size": np.nan,
                     "head_rot_deg": np.nan, "time_unc_s": WIN_S, **side_stats(a)})
events = pd.DataFrame(rows).sort_values("time").reset_index(drop=True)

##-----segments-----##
t_start, t_end = tc.index[0], tc.index[-1] + pd.Timedelta(seconds=WIN_S)
bounds = sorted({t_start, t_end, *ev_head.time, *ev_tilt.time,
                 *[t for v in stretch.values() for ab in v for t in ab]})
# boundaries less than an hour apart (e.g. a step at the edge of a stretch) are merged
merged = [bounds[0]]
for t in bounds[1:]:
    if t - merged[-1] >= pd.Timedelta(hours=1):
        merged.append(t)
merged[-1] = t_end
bounds = merged
seg_rows = []
for a, b in zip(bounds[:-1], bounds[1:]):
    w = (tc.index >= a) & (tc.index < b)
    if w.sum() < 2:
        continue
    fl = flags.reindex(tc.index)[w].fillna(False)
    state = "unsteady" if fl.unsteady.mean() > 0.5 else "low_coh" if fl.low_coh.mean() > 0.5 else "stable"
    seg_rows.append({"sensor": sensor_id, "start": a, "end": b, "hours": (b - a) / pd.Timedelta(hours=1),
                     "state": state, "n_win": int(w.sum()),
                     "dtheta_deg": circ_loc(dth_c.reindex(tc.index)[w]),
                     "dtheta_spread_deg": np.nanmedian(np.abs(wrap(dth_c.reindex(tc.index)[w] - circ_loc(dth_c.reindex(tc.index)[w])))),
                     "tilt_deg": tc.tilt_deg[w].median(),
                     "tilt_bearing_deg": circ_loc(tc.tilt_bearing[w]) % 360,
                     "coh_w_dpdt": tc.coh_w_dpdt[w].median(),
                     "ref_coh": ref_coh.reindex(tc.index)[w].median()})
segments = pd.DataFrame(seg_rows)
segments["head_rot_vs_start_deg"] = -wrap(segments.dtheta_deg - segments.dtheta_deg.iloc[0])

events.to_csv(out_dir / f"{sensor_id}_headmotion_events.csv", index=False, float_format="%.3f")
segments.to_csv(out_dir / f"{sensor_id}_headmotion_segments.csv", index=False, float_format="%.3f")
with pd.option_context("display.width", 200, "display.max_columns", 20):
    print(events.drop(columns="sensor").round(2).to_string(index=False))
    print()
    print(segments.drop(columns="sensor").round(2).to_string(index=False))

### 5. Figures
#### 5a. Overview
Shared time axis. Solid vertical lines = heading steps, dotted = tilt steps; shaded = disturbed
stretches (orange `unsteady`, purple `low_coh`).
1. Δθ against the reference mean (blue) and against each reference (grays). The light band is the
   control pair's own Δθ (its median removed, re-centred on the target's first-day level): the spread
   refraction and swell changes give a fixed head.
2. Step statistic, target and control, with the threshold.
3. w-dp/dt coherence, 4. tilt, 5. Hs_p, 6. % of samples failing the correlation gate,
7. median beam correlation, 8. median beam amplitude, 9. canister heading/pitch/roll anomalies (the
   canister should **not** move when only the head does).

In [ ]:
def mark_events(ax, show_label=False):
    for k, v in stretch.items():
        for a, b in v:
            ax.axvspan(a, b, color=SHADE[k], alpha=0.6, lw=0, zorder=0)
    for t in ev_head.time:
        ax.axvline(t, color=INK, lw=0.9, zorder=1)
    for t in ev_tilt.time:
        ax.axvline(t, color=INK2, lw=0.9, ls=":", zorder=1)

def centred(x, c):
    """Wrap a deg series to within +-180 of c so it plots without 360 jumps."""
    return wrap(x - c) + c

first_day = dth_c[dth_c.index < dth_c.dropna().index[0] + pd.Timedelta(days=1)]
c0 = circ_loc(first_day)
fig, axs = plt.subplots(9, 1, figsize=(12, 19), sharex=True, constrained_layout=True)

ax = axs[0]
ctrl_band = centred(ctrl_dth - circ_loc(ctrl_dth) + c0, c0).rolling(nC, center=True, min_periods=nC // 3)
ax.fill_between(ctrl_band.median().index, ctrl_band.quantile(0.1), ctrl_band.quantile(0.9),
                color="#d3d1cb", lw=0, label=f"control {CONTROL_PAIR[0]}−{CONTROL_PAIR[1]} 10-90 % (shifted)")
for r, col in zip(refs, REF_COLORS):
    ax.plot(dth_c_ref[r].index, centred(dth_c_ref[r], c0), ".", ms=1.5, color=col, label=f"vs {r}")
ax.plot(dth_c.index, centred(dth_c, c0), ".", ms=2.5, color=SERIES, label="vs reference mean")
for _, e in ev_head.iterrows():
    ax.annotate(f"{-e.step:+.0f}°", (e.time, centred(e.after, c0)), xytext=(4, 12), textcoords="offset points",
                color=INK, fontsize=9, fontweight="bold")
ax.set_ylabel("Δθ (deg)")
ax.set_title(f"{sensor_id}: wave direction minus fixed-head references ({WIN_S}-s windows); labels = head rotation "
             f"(+ clockwise)", loc="left", color=INK)
ax.legend(frameon=False, loc="upper left", ncol=4, markerscale=4)

ax = axs[1]
ax.plot(ctrl_s.index, ctrl_s, color=REF_COLORS[1], lw=0.8, label="control")
ax.plot(s_head.index, s_head, color=SERIES, lw=1, label=sensor_id)
for y in (-STEP_THR, STEP_THR):
    ax.axhline(y, color=INK2, ls="--", lw=0.8)
ax.set_ylabel("deg")
ax.set_title(f"heading step statistic (next {STEP_HALF_H} h − previous {STEP_HALF_H} h); dashed = ±{STEP_THR:.1f}",
             loc="left", color=INK)
ax.legend(frameon=False, loc="upper left", ncol=2)

ax = axs[2]
for r, col in zip(refs, REF_COLORS):
    ax.plot(diag[r]["c"].index, diag[r]["c"].coh_w_dpdt, ".", ms=1.5, color=col, label=r)
ax.plot(tc.index, tc.coh_w_dpdt, ".", ms=2.5, color=SERIES, label=sensor_id)
ax.set_ylim(-0.2, 1.05)
ax.set_title("coherence of w with dp/dt (≈ +1 for a vertical head)", loc="left", color=INK)
ax.legend(frameon=False, loc="lower left", ncol=3, markerscale=4)

ax = axs[3]
for r, col in zip(refs, REF_COLORS):
    ax.plot(diag[r]["c"].index, diag[r]["c"].tilt_deg, ".", ms=1.5, color=col, label=r)
ax.plot(tc.index, tc.tilt_deg, ".", ms=2.5, color=SERIES, label=sensor_id)
ax.set_ylabel("deg")
ax.set_ylim(0, np.nanpercentile(tc.tilt_deg, 99.5) * 1.2)
ax.set_title("apparent head tilt (in-phase w-u leakage; includes bed slope and wave nonlinearity)", loc="left", color=INK)
ax.legend(frameon=False, loc="upper left", ncol=3, markerscale=4)

ax = axs[4]
ax.plot(tc.index, tc.Hs_p, color=SERIES, lw=0.8)
ax.set_ylabel("m")
ax.set_title("Hs_p, wave band (bed pressure, not depth-corrected)", loc="left", color=INK)

ax = axs[5]
ax.plot(tc.index, tc.gated_pct, color=SERIES, lw=0.8)
ax.set_ylabel("%")
ax.set_yscale("symlog", linthresh=1)
ax.set_title("samples failing the correlation gate", loc="left", color=INK)

for ax, var, lab in [(axs[6], "corr", "median beam correlation (%)"), (axs[7], "amp", "median beam amplitude (counts)")]:
    for i in range(3):
        ax.plot(tc.index, tc[f"{var}{i + 1}"], color=BEAM_COLORS[i], lw=0.8, label=f"beam {i + 1}")
    ax.set_title(lab, loc="left", color=INK)
    ax.legend(frameon=False, loc="lower left", ncol=3)

ax = axs[8]
for v, col in zip(["can_heading", "can_pitch", "can_roll"], BEAM_COLORS):
    x = tc[v]
    anom = wrap(x - circ_loc(x)) if v == "can_heading" else x - x.median()
    ax.plot(tc.index, anom, color=col, lw=0.8, label=v.replace("can_", ""))
ax.set_ylabel("deg")
ax.set_title("canister heading / pitch / roll, anomaly from the record median (should stay flat)", loc="left", color=INK)
ax.legend(frameon=False, loc="upper left", ncol=3)

for ax in axs:
    mark_events(ax)
axs[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
fig.savefig(fig_dir / f"{sensor_id}_headmotion.png", dpi=150)
plt.show()

#### 5b. Event zooms
±`ZOOM_H` around each heading or tilt step, on the fine windows and the 2 Hz record:
Δθ with the fitted levels, coherence (target and reference mean), tilt, the 1-min **minimum**
correlation per beam (a knock or obstruction shows up as a dropout), 1-min median amplitude, and the
canister attitude.

In [ ]:
t_raw = pd.DatetimeIndex(raw["t"])
ref_coh_f = pd.concat([diag[r]["f"].coh_w_dpdt for r in refs], axis=1).mean(1)
steps = pd.concat([ev_head.assign(kind="heading"), ev_tilt.assign(kind="tilt")]).sort_values("time")
for i, (_, e) in enumerate(steps.iterrows()):
    a, b = e.time - pd.Timedelta(hours=ZOOM_H), e.time + pd.Timedelta(hours=ZOOM_H)
    fw = (tf.index >= a) & (tf.index < b)
    i0, i1 = t_raw.searchsorted(a), t_raw.searchsorted(b)
    fig, axs = plt.subplots(6, 1, figsize=(11, 12), sharex=True, constrained_layout=True)

    ax = axs[0]
    x = dth_f[(dth_f.index >= a) & (dth_f.index < b)]
    cz = circ_loc(x)
    ax.plot(x.index, centred(x, cz), ".", ms=3, color=SERIES)
    if e.kind == "heading":
        ax.hlines([centred(e.before, cz), centred(e.after, cz)], [a, e.time], [e.time, b], color=INK, lw=1.2)
    ax.set_ylabel("Δθ (deg)")
    title = (f"head rotation {-e.step:+.1f}° (+ clockwise)" if e.kind == "heading" else f"tilt step {e.step:+.1f}°")
    ax.set_title(f"{sensor_id}: {e.kind} step at {e.time:%Y-%m-%d %H:%M} (± {FINE_WIN_S // 60} min); {title}",
                 loc="left", color=INK)

    ax = axs[1]
    ax.plot(ref_coh_f[(ref_coh_f.index >= a) & (ref_coh_f.index < b)], ".", ms=2, color=REF_COLORS[0], label="reference mean")
    ax.plot(tf.index[fw], tf.coh_w_dpdt[fw], ".", ms=3, color=SERIES, label=sensor_id)
    ax.set_ylim(-0.2, 1.05)
    ax.set_title(f"coherence of w with dp/dt ({FINE_WIN_S}-s windows)", loc="left", color=INK)
    ax.legend(frameon=False, loc="lower left", ncol=2, markerscale=3)

    ax = axs[2]
    ax.plot(tf.index[fw], tf.tilt_deg[fw], ".", ms=3, color=SERIES)
    if e.kind == "tilt":
        ax.hlines([e.before, e.after], [a, e.time], [e.time, b], color=INK, lw=1.2)
    ax.set_ylabel("deg")
    ax.set_title("apparent head tilt", loc="left", color=INK)

    sl = slice(i0, i1)
    for ax, var, agg, lab in [(axs[3], "corr", "min", "1-min minimum beam correlation (%)"),
                              (axs[4], "amp", "median", "1-min median beam amplitude (counts)")]:
        for j in range(3):
            y = pd.Series(raw[var][j, sl].astype(float), index=t_raw[sl]).resample("1min").agg(agg)
            ax.plot(y.index, y, color=BEAM_COLORS[j], lw=0.8, label=f"beam {j + 1}")
        ax.set_title(lab, loc="left", color=INK)
        ax.legend(frameon=False, loc="lower left", ncol=3)

    ax = axs[5]
    for v, col in zip(["heading", "pitch", "roll"], BEAM_COLORS):
        y = pd.Series(raw[v][sl].astype(float), index=t_raw[sl]).resample("1min").median()
        ax.plot(y.index, wrap(y - circ_loc(y)) if v == "heading" else y - y.median(), color=col, lw=0.8, label=v)
    ax.set_ylabel("deg")
    ax.set_title("canister attitude, 1-min median anomaly", loc="left", color=INK)
    ax.legend(frameon=False, loc="upper left", ncol=3)

    for ax in axs:
        ax.axvline(e.time, color=INK if e.kind == "heading" else INK2, lw=0.9, ls="-" if e.kind == "heading" else ":")
    axs[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    fig.savefig(fig_dir / f"{sensor_id}_headmotion_event{i + 1}.png", dpi=130)
    plt.show()
if steps.empty:
    print(f"{sensor_id}: no heading or tilt steps above threshold")

#### 5c. Tilt vs wave height
Nonlinear waves put some w in phase with u, more so in big swell and shallow water. If the target's
tilt rises with Hs the way the references' does, a tilt "step" during a swell event is probably not the
head. A real tilt change shows as a jump between segments at the same Hs.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
for r, col in zip(refs, REF_COLORS):
    ax.plot(diag[r]["c"].Hs_p, diag[r]["c"].tilt_deg, ".", ms=2, color=col, label=r)
days = (tc.index - tc.index[0]) / pd.Timedelta(days=1)
sc = ax.scatter(tc.Hs_p, tc.tilt_deg, c=days, s=4, cmap="Blues", vmin=-10)
ax.plot([], [], ".", color=SERIES, label=f"{sensor_id} (shade = time)")   # legend proxy
fig.colorbar(sc, ax=ax, label=f"days since {tc.index[0]:%m-%d}")
ax.set_xlabel("Hs_p (m, bed pressure)")
ax.set_ylabel("apparent tilt (deg)")
ax.set_ylim(0, np.nanpercentile(tc.tilt_deg, 99.5) * 1.2)
ax.set_title(f"{sensor_id}: apparent tilt vs wave height ({WIN_S}-s windows)", loc="left", color=INK)
ax.legend(frameon=False, loc="upper left", markerscale=4)
fig.savefig(fig_dir / f"{sensor_id}_headmotion_tilt_hs.png", dpi=150)
plt.show()

#### 5d. All loose-head sensors
Reads every `{S}_headmotion_segments.csv` in `TARGETS` written so far (run the batch cell first for the
full set). Bars: gray = stable, orange = unsteady, purple = low coherence; the label is the head
rotation since the start of the record (+ clockwise from above). Ticks = heading steps.

In [ ]:
have = [S for S in TARGETS if (out_dir / f"{S}_headmotion_segments.csv").exists()]
fig, ax = plt.subplots(figsize=(12, 0.9 + 0.8 * len(have)), constrained_layout=True)
STATE_COLOR = {"stable": GRID, **SHADE}
for y, S in enumerate(have[::-1]):
    sg = pd.read_csv(out_dir / f"{S}_headmotion_segments.csv", parse_dates=["start", "end"])
    ev = pd.read_csv(out_dir / f"{S}_headmotion_events.csv", parse_dates=["time", "end"])
    for _, r in sg.iterrows():
        ax.barh(y, r.end - r.start, left=r.start, height=0.6, color=STATE_COLOR[r.state], edgecolor="white", lw=1)
        if r.hours >= 24:
            v = int(round(r.head_rot_vs_start_deg))
            ax.text(r.start + (r.end - r.start) / 2, y, f"{v:+d}°" if v else "0°", ha="center",
                    va="center", fontsize=7, color=INK)
    for t in ev.time[ev.type == "heading_step"]:
        ax.plot([t, t], [y - 0.38, y + 0.38], color=INK, lw=1.2)
ax.set_yticks(range(len(have)), have[::-1])
ax.grid(axis="y", visible=False)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in STATE_COLOR.values()]
ax.legend(handles, list(STATE_COLOR), frameon=False, loc="upper left", bbox_to_anchor=(0, -0.12), ncol=3)
ax.set_title("loose-head sensors: head state and rotation since deployment (+ clockwise from above)",
             loc="left", color=INK)
fig.savefig(fig_dir / "headmotion_timeline.png", dpi=150)
plt.show()

### Run the other sensors
Off by default. Set `RUN_BATCH = True` and run this cell on its own (save the notebook first: the cell
reads it from disk). Each sensor in `TARGETS` runs in a fresh kernel with `sensor_id` overridden; the
executed copy goes to `processing/qc_runs/adv_head_motion_{S}.ipynb`. Re-run 5d afterwards for the
combined timeline. Any other Vector (e.g. a fixed head as a null check) can be added to the list.

In [ ]:
##-----batch: run this notebook once per sensor in TARGETS-----##
RUN_BATCH = False

if RUN_BATCH:
    import nbformat
    from nbclient import NotebookClient
    nb_path = root / "processing" / "adv_head_motion.ipynb"
    run_dir = root / "processing" / "qc_runs"
    run_dir.mkdir(exist_ok=True)
    for sid in TARGETS:
        nb = nbformat.read(nb_path, as_version=4)
        nb.cells = [c for c in nb.cells if "batch" not in c.metadata.get("tags", [])]
        par = next(c for c in nb.cells if "parameters" in c.metadata.get("tags", []))
        par.source += f'\nsensor_id = "{sid}"   # injected by the batch cell'
        print(f"{sid}: running ...", flush=True)
        try:
            NotebookClient(nb, timeout=None, kernel_name="dolf_env",
                           resources={"metadata": {"path": str(nb_path.parent)}}).execute()
            print(f"{sid}: ok")
        except Exception as e:   # keep going; the error is in the saved copy
            print(f"{sid}: FAILED ({type(e).__name__}), see the saved copy")
        nbformat.write(nb, run_dir / f"adv_head_motion_{sid}.ipynb")